# Neural Network Month · Project 2
## Notebook 2 of 3 — The CNN

In Project 1 every neuron looked at every input. A photo or a scan has hundreds of thousands of pixels, and that approach would drown in parameters. A **convolutional neural network (CNN)** fixes this with one idea: slide a small set of learned **filters** across the image, so that the same few detectors (edges, blobs, textures) are reused everywhere.

Here is how the network we build today works:

- **Convolution:** a small filter slides across the image and measures how strongly each patch matches a pattern. Many filters give many **feature maps**.
- **Stride:** moving the filter in bigger steps shrinks the map, so each later layer sees a wider region of the scan.
- **Residual shortcut:** each block adds its input back to its output. That keeps learning stable in deep networks.
- **Batch normalisation:** rescales activations so training is steadier.
- **Global average pooling:** collapses each final feature map to one number, giving 256 numbers per scan, then a final layer turns them into class scores.

We keep the **backpropagation** idea from Project 1; PyTorch just does the chain rule for us. This time we use a framework instead of writing it by hand.

**This notebook:** build the CNN, train it on one fold, and look inside it. Two figures.

**Before you run:** switch Colab to a GPU (Runtime → Change runtime type → T4 GPU). Everything also runs on CPU with `SMOKE_TEST = True`, just slowly.

## 1 · Setup, data, model code

In [ ]:
import os, glob, zipfile, hashlib, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedKFold

# One palette for the whole series. Each tumor class keeps its colour in every figure.
P = {
    "glioma":     "#C97B4A",  # clay
    "meningioma": "#2F6F73",  # teal
    "pituitary":  "#D9A441",  # ochre
    "no_tumor":   "#7A9E7E",  # sage
    "ink":        "#4A3F35",  # bark   - text, validation lines
    "paper":      "#F5EFE6",  # sand   - background
    "grid":       "#D8CFC0",  # stone  - gridlines
    "stone":      "#A89F91",  # neutral bars and training lines
    "rust":       "#A5402D",  # warning colour (duplicates, errors)
}
CLASS_COLORS = {"Glioma": P["glioma"], "Meningioma": P["meningioma"],
                "Pituitary": P["pituitary"], "No Tumor": P["no_tumor"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["ink"]])
CAM_CMAP = LinearSegmentedColormap.from_list("cam", [P["pituitary"], P["glioma"], P["rust"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

In [ ]:
# Where is the data? The cell tries, in order: an already-extracted folder, your uploaded
# Kaggle zip, then a Kaggle download. Easiest route: download archive.zip from the Kaggle page
# and upload it to /content (the Colab file panel), or point ZIP_PATH at it in Google Drive.
ZIP_PATH  = "/content/archive.zip"
DATA_DIR  = "mri_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/Train", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        path = kagglehub.dataset_download("ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes")
        root = locate_root(path)
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Upload the Kaggle zip to Colab and set ZIP_PATH, or add Kaggle credentials.")
print("Data folder:", root)

In [ ]:
# ---- Cleaning settings (the only places we touch the data) ----
KEEP_ONLY_512   = True   # keep only 512x512 scans, so nothing is ever resized
MIN_CLASS_COUNT = 100    # a class with fewer images than this after cleaning is dropped
N_FOLDS, SEED   = 5, 42

def index_images(root):
    """One row per image: where it is, its shipped split, class, size, and a fingerprint of its bytes."""
    rows = []
    for p in sorted(glob.glob(f"{root}/*/*/images/*.jpg")):
        rel = os.path.relpath(p, root)
        split, cls = rel.split(os.sep)[:2]
        with Image.open(p) as im:
            w, h = im.size
        with open(p, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        rows.append((rel, split, cls, w, h, md5))
    return pd.DataFrame(rows, columns=["path", "shipped_split", "cls", "w", "h", "md5"])

def clean(raw):
    stages = {"Raw Download": raw}
    df = raw
    if KEEP_ONLY_512:
        df = df[(df.w == 512) & (df.h == 512)]
    stages["512 Pixels Only" if KEEP_ONLY_512 else "All Sizes Kept"] = df

    # How contaminated is the shipped validation set? (measured before removing anything else)
    train_hashes = set(df[df.shipped_split == "Train"].md5)
    val = df[df.shipped_split == "Val"]
    leak = {"val_total": len(val), "val_copies": int(val.md5.isin(train_hashes).sum()),
            "all_total": len(df), "all_redundant": int(len(df) - df.md5.nunique())}

    # Drop any file whose identical twin carries a different label, then keep one copy of each image
    mixed = df.groupby("md5")["cls"].nunique()
    leak["conflicting_labels"] = int((mixed > 1).sum())
    df = df[~df.md5.isin(mixed[mixed > 1].index)]
    df = df.sort_values(["shipped_split", "path"]).drop_duplicates("md5", keep="first")
    stages["Duplicates Removed"] = df

    counts = df.cls.value_counts()
    dropped = {c: int(n) for c, n in counts.items() if n < MIN_CLASS_COUNT}
    df = df[~df.cls.isin(list(dropped))]
    stages["Final Dataset"] = df

    df = df.reset_index(drop=True)
    classes = sorted(df.cls.unique())
    df["label"] = df.cls.map({c: i for i, c in enumerate(classes)})
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, test_idx) in enumerate(skf.split(df, df.label)):
        df.loc[test_idx, "fold"] = k
    return df, stages, leak, dropped, classes

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device)
if device.type == "cpu":
    print("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU. "
          "(SMOKE_TEST = True still works on CPU, just slowly.)")

## 2 · Load the cleaned scan list

If you ran Notebook 1 in this session, `clean_manifest.csv` is reused. Otherwise it is rebuilt with identical settings.

In [ ]:
SMOKE_TEST = False   # True = quick dry run (20 scans per class, 2 epochs) to check everything works
EPOCHS     = 2 if SMOKE_TEST else 25

if os.path.exists("clean_manifest.csv"):
    manifest = pd.read_csv("clean_manifest.csv")
else:                                   # Notebook 1 was not run in this session: rebuild the same table
    manifest, _, _, _, _ = clean(index_images(root))
if SMOKE_TEST:
    manifest = manifest.groupby("cls", group_keys=False).sample(20, random_state=0).reset_index(drop=True)
    for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(manifest, manifest.label)):
        manifest.loc[te, "fold"] = k        # re-balance the folds on the small subset
classes = sorted(manifest.cls.unique())
n_classes = len(classes)
print(f"{len(manifest)} scans | classes: {classes}")
manifest.groupby("cls").size()

## 3 · Load the scans

Every scan becomes one 512×512 grey channel and stays at **full resolution**. The network's first layer does the downsampling, so we never resize your images ourselves.

In [ ]:
def load_images(df, root):
    """Read every scan as one 512x512 grey channel. The RGB files are grey-valued copies,
    so one channel keeps all of their information. Nothing else is changed."""
    X = np.empty((len(df), 512, 512), dtype=np.uint8)
    for i, p in enumerate(df.path):
        im = Image.open(os.path.join(root, p)).convert("L")
        if im.size != (512, 512):          # only happens if KEEP_ONLY_512 = False
            im = im.resize((512, 512), Image.BICUBIC)
        X[i] = np.asarray(im)
    return X

t0 = time.time()
X = load_images(manifest, root)
labels = manifest.label.values
folds  = manifest.fold.values
print(f"Loaded {X.shape[0]} scans, {X.nbytes/1e9:.2f} GB, in {time.time()-t0:.0f}s")

## 4 · The network

Read `ResBlock` first (the repeating unit), then `BrainCNN` (how the units stack). The first layer is a 4×4 convolution with stride 4: it turns every non-overlapping 4×4 patch into 32 numbers, shrinking 512×512 to 128×128. Each later stage halves the map size and doubles the number of filters.

| Stage | Output shape (channels × height × width) |
|---|---|
| Input scan | 1 × 512 × 512 |
| Stem | 32 × 128 × 128 |
| Stage 1 | 32 × 128 × 128 |
| Stage 2 | 64 × 64 × 64 |
| Stage 3 | 128 × 32 × 32 |
| Stage 4 | 256 × 16 × 16 |
| Pool, dropout, linear | one score per class |

In [ ]:
class ResBlock(nn.Module):
    """Two 3x3 convolutions plus a shortcut that lets the signal skip past them."""
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = (nn.Identity() if stride == 1 and c_in == c_out else
                     nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + self.skip(x))

def stage(c_in, c_out, stride, n_blocks):
    return nn.Sequential(ResBlock(c_in, c_out, stride), *[ResBlock(c_out, c_out, 1) for _ in range(n_blocks - 1)])

class BrainCNN(nn.Module):
    """512x512 scan -> learned 4x downsampling -> four residual stages -> average pool -> class scores."""
    def __init__(self, n_classes, widths=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv2d(1, widths[0], 4, 4, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        self.stage1 = stage(widths[0], widths[0], 1, 1)
        self.stage2 = stage(widths[0], widths[1], 2, 2)
        self.stage3 = stage(widths[1], widths[2], 2, 2)
        self.stage4 = stage(widths[2], widths[3], 2, 2)
        self.drop, self.fc = nn.Dropout(dropout), nn.Linear(widths[3], n_classes)

    def features(self, x):                    # the last feature maps, before pooling
        return self.stage4(self.stage3(self.stage2(self.stage1(self.stem(x)))))

    def forward(self, x):
        f = self.features(x)
        return self.fc(self.drop(f.mean(dim=(2, 3))))

In [ ]:
print("Trainable parameters:", f"{sum(p.numel() for p in BrainCNN(n_classes).parameters()):,}")

## 5 · Training code

`train_fold` trains on every scan outside a fold and tests on the scans inside it, using:

- **AdamW** (gradient descent with momentum and an adaptive step size) and a **one-cycle learning-rate schedule**: warm up, then glide down.
- **Light augmentation during training only**: random left-right flips, small contrast and brightness changes, and small shifts. This is the "make the model robust to the mess" part. The image files and the test folds are never touched.
- **Dropout**, which randomly silences some of the 256 pooled numbers each step so the network cannot lean on any single one.

The model from the final epoch is the one we keep. We do not pick the best epoch by peeking at the test fold, which would flatter the score.

In [ ]:
def augment(xb):
    """Light, train-time-only variation (the files on disk are never changed): flip, contrast/brightness, small shift."""
    n, dev = xb.size(0), xb.device
    flip = torch.rand(n, device=dev) < 0.5
    xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
    c = 1 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.3
    b = (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.1
    xb = ((xb - 0.5) * c + 0.5 + b).clamp(0, 1)
    dx, dy = torch.randint(-16, 17, (2,)).tolist()
    return torch.roll(xb, shifts=(dy, dx), dims=(2, 3))

@torch.no_grad()
def predict(model, Xg, idx, bs=64):
    model.eval()
    out = []
    for i in range(0, len(idx), bs):
        xb = Xg[idx[i:i + bs]].unsqueeze(1).float().div(255)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(model(xb).float().cpu())
    return torch.cat(out)

def train_fold(Xg, yg, tr_idx, va_idx, epochs, lr=2e-3, bs=32, use_augment=True, seed=0, verbose=True):
    torch.manual_seed(seed)
    model = BrainCNN(n_classes).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(tr_idx) / bs))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    yv = yg[va_idx].cpu()
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for ep in range(epochs):
        model.train(); t0 = time.time()
        perm = tr_idx[torch.randperm(len(tr_idx), device=tr_idx.device)]
        loss_sum = correct = 0.0
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            xb, yb = Xg[b].unsqueeze(1).float().div(255), yg[b]
            if use_augment:
                xb = augment(xb)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item() * len(b); correct += (logits.argmax(1) == yb).sum().item()
        lv = predict(model, Xg, va_idx)
        hist["train_loss"].append(loss_sum / len(tr_idx)); hist["train_acc"].append(correct / len(tr_idx))
        hist["val_loss"].append(F.cross_entropy(lv, yv).item()); hist["val_acc"].append((lv.argmax(1) == yv).float().mean().item())
        if verbose and (ep % 5 == 4 or ep == epochs - 1 or ep == 0):
            print(f"epoch {ep+1:>2}/{epochs}  train acc {hist['train_acc'][-1]:.3f}  "
                  f"val acc {hist['val_acc'][-1]:.3f}  ({time.time()-t0:.0f}s/epoch)")
    return model, hist, lv

## 6 · Train on one fold

Fold 1 is held out; the other four folds are the training data.

In [ ]:
Xg = torch.from_numpy(X).to(device)
yg = torch.as_tensor(np.array(labels), dtype=torch.long, device=device)
tr_idx = torch.as_tensor(np.where(folds != 0)[0], device=device)
va_idx = torch.as_tensor(np.where(folds == 0)[0], device=device)
print(f"Train scans: {len(tr_idx)} | held-out scans: {len(va_idx)}")

model, hist, logits_va = train_fold(Xg, yg, tr_idx, va_idx, epochs=EPOCHS)
print(f"\nHeld-out accuracy on fold 1: {hist['val_acc'][-1]:.3f}")

## 7 · Figure 5 — learning curves

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, key, yl in [(a1, "loss", "Cross-entropy loss"), (a2, "acc", "Accuracy")]:
    ax.plot(ep, hist[f"train_{key}"], color=P["stone"], linewidth=2.6, label="Training")
    ax.plot(ep, hist[f"val_{key}"], color=P["ink"], linewidth=2.2, linestyle="--", label="Held-out fold")
    ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.legend(frameon=False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
a1.set_title("Loss"); a2.set_title("Accuracy")
fig.suptitle("Training and Validation Curves", y=1.02)
fig.tight_layout(); save_fig(fig, "05_training_curves"); plt.show()

**How to read it.** The training curve is measured *while* the network is being randomly perturbed (augmentation and dropout), so it can sit below the held-out curve early on. That is normal here. The warning sign would be the held-out loss rising while the training loss keeps falling, which is overfitting.

## 8 · Figure 6 — what the CNN sees

We pass one held-out scan per class through the trained network and photograph the *inside* at each stage. Each panel averages that stage's feature maps over their channels, so bright areas are where the stage's filters respond most strongly. Notice the picture getting **smaller and more abstract**: early stages keep fine detail, and later stages hold a coarse map of "where the interesting stuff is".

In [ ]:
@torch.no_grad()
def stage_maps(model, x):
    h = model.stem(x); out = [("After Stem", h)]
    for name, title in [("stage1", "Stage One"), ("stage2", "Stage Two"), ("stage3", "Stage Three"), ("stage4", "Stage Four")]:
        h = getattr(model, name)(h); out.append((title, h))
    return out

model.eval()
va_np = va_idx.cpu().numpy()
rows = []
for c in range(n_classes):
    cand = va_np[labels[va_np] == c]
    if len(cand): rows.append((classes[c], cand[0]))

fig, axes = plt.subplots(len(rows), 6, figsize=(15, 2.7 * len(rows)))
axes = np.atleast_2d(axes)
for r, (cname, i) in enumerate(rows):
    x = Xg[i][None, None].float().div(255)
    maps = stage_maps(model, x)
    axes[r][0].imshow(X[i], cmap="gray"); axes[r][0].set_xlabel("1 × 512 × 512", fontsize=9)
    if r == 0: axes[r][0].set_title("Input Scan")
    axes[r][0].text(-0.08, 0.5, cname, transform=axes[r][0].transAxes, rotation=90, ha="right", va="center",
                    fontsize=13, fontweight="bold", color=CLASS_COLORS[cname])
    for j, (title, h) in enumerate(maps, start=1):
        axes[r][j].imshow(h[0].mean(0).cpu().numpy(), cmap=SEQ_CMAP)
        axes[r][j].set_xlabel(f"{h.shape[1]} × {h.shape[2]} × {h.shape[3]}", fontsize=9)
        if r == 0: axes[r][j].set_title(title)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        for s in a.spines.values(): s.set_visible(False)
fig.suptitle("How The CNN Sees", y=1.0)
fig.tight_layout(); save_fig(fig, "06_cnn_feature_maps"); plt.show()

The labels under each panel are the real tensor shapes (channels × height × width). Four steps of striding take a 512×512 scan down to a 16×16 grid, where each cell summarises a large patch of the original image through a stack of filters. The final layer reads those 256 summary numbers and makes the call.

**One caution about the picture:** these maps show where filters *fire*, not a verdict on what is medically meaningful. Notebook 3 asks the sharper question: where does the network look when it decides?

**Next:** Notebook 3 runs the full stratified 5-fold validation.

In [ ]:
# Optional: download this notebook's figures
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb2_figures", "zip", "figures")
    files.download("nb2_figures.zip")
except ImportError:
    print("Figures are saved in the 'figures' folder.")